In [1]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 07. 역전파 수학 — 체인룰의 기계적 적용 ⭐

> **학습 목표**
> - 연쇄 법칙이 신경망 학습에 어떻게 적용되는지 유도한다
> - 계산 그래프(computational graph)를 그리고 국소 도함수를 계산한다
> - Autograd를 직접 구현해 본다
> - 역전파에서 CPU vs GPU 속도 차이를 체감한다

이 장은 딥러닝 교육의 심장이다. PyTorch의 `autograd`가 내부에서 무엇을 하는지 직접 구현해 본다.

## 7.1 왜 역전파인가

신경망의 손실 $\mathcal{L}(\theta)$를 최소화하려면 그래디언트 $\nabla_\theta \mathcal{L}$가 필요하다.

**순방향 수치 미분(forward-mode AD)**: 각 파라미터마다 한 번씩 순전파.
- 파라미터 $N$개 → $N$번 순전파. 느리다.

**역방향 자동 미분(reverse-mode AD, 역전파)**: 한 번의 역전파로 모든 파라미터의 그래디언트 계산.
- $N$개 파라미터 → 1번 순전파 + 1번 역전파. 압도적으로 빠르다.


## 7.2 연쇄 법칙 복습

$$\frac{d}{dx} f(g(x)) = f'(g(x)) \cdot g'(x)$$

다변수로 확장: $y = f(u_1, u_2, \ldots, u_n)$, $u_i = g_i(x)$일 때

$$\frac{dy}{dx} = \sum_i \frac{\partial f}{\partial u_i} \cdot \frac{du_i}{dx}$$

이것이 역전파의 수학적 핵심.


In [2]:
import numpy as np
import matplotlib.pyplot as plt

# 간단한 합성 함수: y = f(g(h(x))) = (h(x)^2 + 1) * 2, h(x) = x + 1
# = ((x+1)^2 + 1) * 2
def f(x): return ((x + 1)**2 + 1) * 2
def df_dx(x): return 2 * (x + 1) * 2 * 1  # 연쇄 법칙

# 수치 미분으로 검증
def numerical_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

x0 = 3.0
print(f"f(x) = ((x+1)^2 + 1) * 2")
print(f"f'({x0}) 해석적 (연쇄 법칙): {df_dx(x0)}")
print(f"f'({x0}) 수치: {numerical_diff(f, x0):.6f}")
print(f"Error: {abs(df_dx(x0) - numerical_diff(f, x0)):.2e}")


f(x) = ((x+1)^2 + 1) * 2
f'(3.0) 해석적 (연쇄 법칙): 16.0
f'(3.0) 수치: 16.000000
Error: 2.50e-10


## 7.3 계산 그래프

복잡한 함수를 기본 연산(덧셈, 곱셈, 활성화 등)의 그래프로 분해.

예: $\mathcal{L} = (\sigma(w \cdot x + b) - y)^2$

```
x ──┐
    ├─→ (×) ──→ (+) ──→ (σ) ──→ (-) ──→ (²) ──→ L
w ──┘         ↑              ↑
              b              y
```

각 노드는 **국소 도함수(local gradient)**만 계산하면 된다. 역전파는 체인룰로 이들을 곱해가는 과정.


In [3]:
# 미니 Autograd 엔진 구현
class Tensor:
    """간단한 자동 Differential Tensor."""
    def __init__(self, data, requires_grad=False, _children=(), _op=''):
        self.data = np.asarray(data, dtype=float)
        self.requires_grad = requires_grad
        self.grad = None
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Tensor({self.data}, grad={self.grad})"

    @property
    def shape(self):
        return self.data.shape

    # Operation 정의
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, _children=(self, other), _op='+')

        def _backward():
            # Addition의 국소 Derivative = 1
            self.grad = (self.grad if self.grad is not None else 0) + out.grad
            other.grad = (other.grad if other.grad is not None else 0) + out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, _children=(self, other), _op='*')

        def _backward():
            # Multiplication의 국소 Derivative: d/da(a*b) = b, d/db(a*b) = a
            self.grad = (self.grad if self.grad is not None else 0) + other.data * out.grad
            other.grad = (other.grad if other.grad is not None else 0) + self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), _children=(self,), _op='relu')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1 / (1 + np.exp(-self.data))
        out = Tensor(s, _children=(self,), _op='sigmoid')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + s * (1 - s) * out.grad
        out._backward = _backward
        return out

    def sum(self):
        out = Tensor(self.data.sum(), _children=(self,), _op='sum')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # 위상 Alignment
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        self.grad = np.ones_like(self.data)  # dL/dL = 1
        for v in reversed(topo):
            v._backward()

# Test: f(x, y) = (x + y) * (x - y) = x^2 - y^2 (간단히 x*x - y*y)
x = Tensor(3.0, requires_grad=True)
y = Tensor(4.0, requires_grad=True)
# z = x*x - y*y  (subtraction = add negative; Implementation 단순화를 위해 직접)
a = x * x  # x^2
b = y * y  # y^2
z = a + (b * Tensor(-1.0))  # x^2 - y^2
z.backward()

print(f"x = {x.data}, y = {y.data}")
print(f"z = x^2 - y^2 = {z.data}")
print(f"dz/dx = {x.grad} (Theory 2x = {2*x.data})")
print(f"dz/dy = {y.grad} (Theory -2y = {-2*y.data})")


x = 3.0, y = 4.0
z = x^2 - y^2 = -7.0
dz/dx = 6.0 (Theory 2x = 6.0)
dz/dy = -8.0 (Theory -2y = -8.0)


## 7.4 선형층 역전파 유도

선형층: $\mathbf{y} = W\mathbf{x} + \mathbf{b}$

손실 $\mathcal{L}$에 대한 그래디언트:

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} \cdot \mathbf{x}^\top = \boldsymbol{\delta} \, \mathbf{x}^\top$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = W^\top \boldsymbol{\delta}$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{b}} = \boldsymbol{\delta}$$

여기서 $\boldsymbol{\delta} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}}$는 "업스트림 그래디언트".


In [4]:
# 선형층 역전파 검증 (PyTorch와 비교)
import torch

# 입력
x_np = np.random.randn(3, 4)  # batch=3, in=4
W_np = np.random.randn(4, 5)  # in=4, out=5
b_np = np.random.randn(5)

# NumPy로 순전파 + 역전파
def linear_forward(x, W, b):
    return x @ W + b

def linear_backward(x, W, delta):
    """delta: dL/dy (N, out)"""
    dW = x.T @ delta          # (in, out)
    db = delta.sum(axis=0)    # (out,)
    dx = delta @ W.T          # (N, in)
    return dW, db, dx

# 임의의 delta (dL/dy)
delta_np = np.random.randn(3, 5)
y_np = linear_forward(x_np, W_np, b_np)
dW_np, db_np, dx_np = linear_backward(x_np, W_np, delta_np)

# PyTorch로 Validation
x_t = torch.tensor(x_np, requires_grad=True)
W_t = torch.tensor(W_np, requires_grad=True)
b_t = torch.tensor(b_np, requires_grad=True)
y_t = x_t @ W_t + b_t
y_t.backward(torch.tensor(delta_np))

print("NumPy vs PyTorch 역전파 비교:")
print(f"  dW Maximum Error: {np.max(np.abs(dW_np - W_t.grad.numpy())):.2e}")
print(f"  db Maximum Error: {np.max(np.abs(db_np - b_t.grad.numpy())):.2e}")
print(f"  dx Maximum Error: {np.max(np.abs(dx_np - x_t.grad.numpy())):.2e}")
print("\n=> NumPy 직접 Implementation과 PyTorch autograd가 정확히 일치!")


NumPy vs PyTorch 역전파 비교:
  dW Maximum Error: 0.00e+00
  db Maximum Error: 0.00e+00
  dx Maximum Error: 8.88e-16

=> NumPy 직접 Implementation과 PyTorch autograd가 정확히 일치!


## 7.5 MLP 역전파 전체 유도

2층 MLP: $\mathbf{o} = W_2 \sigma(W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$

손실 $\mathcal{L} = \|\mathbf{o} - \mathbf{y}\|^2$ (MSE)

역전파:
1. $\frac{\partial \mathcal{L}}{\partial \mathbf{o}} = 2(\mathbf{o} - \mathbf{y})$
2. $\frac{\partial \mathcal{L}}{\partial W_2} = \frac{\partial \mathcal{L}}{\partial \mathbf{o}} \mathbf{h}^\top$
3. $\frac{\partial \mathcal{L}}{\partial \mathbf{h}} = W_2^\top \frac{\partial \mathcal{L}}{\partial \mathbf{o}}$
4. $\frac{\partial \mathcal{L}}{\partial \mathbf{z}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}} \odot \sigma'(\mathbf{z}_1)$
5. $\frac{\partial \mathcal{L}}{\partial W_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}_1} \mathbf{x}^\top$

요점: **각 층마다 국소 도함수 × 업스트림 그래디언트**를 반복.


In [5]:
# MLP 역전파 수치 미분으로 검증
# f(W1, b1, W2, b2) = ||W2 @ sigmoid(W1 @ x + b1) + b2 - y||^2

def mlp_loss_and_grads(x, W1, b1, W2, b2, y):
    """Forward Pass + Backward Pass (Vector Input, 단일 Sample)."""
    # Forward Pass
    z1 = W1 @ x + b1
    h = 1 / (1 + np.exp(-z1))  # sigmoid
    o = W2 @ h + b2
    diff = o - y
    loss = np.sum(diff**2)
    # Backward Pass
    do = 2 * diff
    dW2 = np.outer(do, h)
    db2 = do
    dh = W2.T @ do
    dz1 = dh * h * (1 - h)
    dW1 = np.outer(dz1, x)
    db1 = dz1
    return loss, dW1, db1, dW2, db2

# 수치 Differential
def numerical_grad(f, param, h=1e-5):
    """f는 loss만 반환하는 Function. param에 대한 수치 Gradient."""
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = param[idx]
        param[idx] = old + h
        loss_plus = f()
        param[idx] = old - h
        loss_minus = f()
        param[idx] = old
        grad[idx] = (loss_plus - loss_minus) / (2 * h)
        it.iternext()
    return grad

# 작은 MLP로 Validation
np.random.seed(0)
x = np.random.randn(3)
y = np.random.randn(2)
W1 = np.random.randn(4, 3)
b1 = np.random.randn(4)
W2 = np.random.randn(2, 4)
b2 = np.random.randn(2)

loss, dW1, db1, dW2, db2 = mlp_loss_and_grads(x, W1, b1, W2, b2, y)

# 수치 Gradient
loss_fn = lambda: mlp_loss_and_grads(x, W1, b1, W2, b2, y)[0]
dW1_num = numerical_grad(loss_fn, W1)
dW2_num = numerical_grad(loss_fn, W2)

print(f"dW1 최대 오차: {np.max(np.abs(dW1 - dW1_num)):.2e}")
print(f"dW2 Maximum Error: {np.max(np.abs(dW2 - dW2_num)):.2e}")
print("=> 해석적 Backward Pass와 수치 Differential이 일치!")


dW1 최대 오차: 4.81e-11
dW2 Maximum Error: 2.04e-11
=> 해석적 Backward Pass와 수치 Differential이 일치!


## 7.6 그래디언트 소실/폭발

시그모이드 도함수 $\sigma'(x) = \sigma(x)(1-\sigma(x)) \leq 0.25$.

깊은 신경망에서 역전파 시 이 작은 값들이 계속 곱해지면 그래디언트가 지수적으로 감소 → **그래디언트 소실(vanishing gradient)**.

해결책:
- ReLU 계열 활성화 (도함수가 0 또는 1)
- 적절한 가중치 초기화 (He, Xavier)
- Residual connection (트랜스포머에서 배움)
- BatchNorm/LayerNorm


In [6]:
# 그래디언트 소실 시뮬레이션
def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

# 깊이에 따른 그래디언트 크기
np.random.seed(0)
depths = [5, 10, 20, 50, 100]
print(f"{'Depth':>6} | {'Mean |grad|':>15} | {'Minimum |grad|':>15}")
print("-" * 45)
for d in depths:
    x = np.random.randn(100)
    grad = np.ones(100)  # dL/dy
    for _ in range(d):
        # Backward Pass 시그모이드 Layer
        z = np.random.randn(100)
        grad = grad * sigmoid_deriv(z)
    print(f"{d:>6} | {np.mean(np.abs(grad)):>15.2e} | {np.min(np.abs(grad)):>15.2e}")

print("\n=> Depth가 깊어질수록 Gradient가 지수적으로 작아진다 (소실)!")
print("ReLU를 쓰Plane Derivative가 0 또는 1이라 이 문제가 크게 완화된다.")


 Depth |     Mean |grad| |  Minimum |grad|
---------------------------------------------
     5 |        3.71e-04 |        2.23e-05
    10 |        1.51e-07 |        1.51e-08
    20 |        2.49e-14 |        5.22e-16
    50 |        6.93e-35 |        2.79e-38
   100 |        2.70e-69 |        2.82e-73

=> Depth가 깊어질수록 Gradient가 지수적으로 작아진다 (소실)!
ReLU를 쓰Plane Derivative가 0 또는 1이라 이 문제가 크게 완화된다.


## 7.7 [CPU/GPU 벤치마크 ②] 역전파 시간 비교

모델 크기별로 순전파+역전파 시간을 CPU vs GPU에서 비교하자.


In [7]:
# 역전파 벤치마크
import torch
import time
from llm_math.bench import time_fn, format_results_table

def make_mlp_and_run(input_dim, hidden, output_dim, n_layers, batch_size=32, device='cpu'):
    """MLP를 만들고 한 Step Forward Pass+Backward Pass."""
    dims = [input_dim] + [hidden] * (n_layers - 1) + [output_dim]
    layers = [torch.nn.Linear(dims[i], dims[i+1]) for i in range(len(dims) - 1)]
    model = torch.nn.Sequential(*layers).to(device)
    x = torch.randn(batch_size, input_dim, device=device)
    y = torch.randn(batch_size, output_dim, device=device)
    loss_fn = torch.nn.MSELoss()
    opt = torch.optim.SGD(model.parameters(), lr=0.01)

    def step():
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        return loss
    return step

# Model Size별 Comparison
configs = [
    ('small',  100, 64, 10, 3),
    ('medium', 100, 256, 10, 5),
    ('large',  100, 512, 10, 10),
    ('xlarge', 100, 1024, 10, 20),
]

print(f"{'Config':>8} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for name, in_d, hid, out_d, nl in configs:
    step_cpu = make_mlp_and_run(in_d, hid, out_d, nl, device='cpu')
    res_cpu = time_fn(step_cpu, device='cpu', warmup=2, repeat=5)
    if torch.cuda.is_available():
        step_gpu = make_mlp_and_run(in_d, hid, out_d, nl, device='cuda')
        res_gpu = time_fn(step_gpu, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{name:>8} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{name:>8} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ GPU 없음. Colab에서 T4 GPU 런타임으로 전환하Plane 가속을 볼 수 있다.")
    print("  특히 큰 Model(xlarge)에서 CPU는 수 초, GPU는 수십 ms가 걸린다.")


  Config |     CPU (ms) |     GPU (ms) |    Speedup
--------------------------------------------------


   small |        0.341 |          N/A |        N/A
  medium |        3.122 |          N/A |        N/A
   large |       13.843 |          N/A |        N/A


  xlarge |       90.379 |          N/A |        N/A

⚠ GPU 없음. Colab에서 T4 GPU 런타임으로 전환하Plane 가속을 볼 수 있다.
  특히 큰 Model(xlarge)에서 CPU는 수 초, GPU는 수십 ms가 걸린다.


## 7.8 핵심 정리

| 개념 | 수식 | 의미 |
|---|---|---|
| 연쇄 법칙 | $\frac{d}{dx}f(g(x)) = f'(g)g'(x)$ | 합성함수 미분 |
| 계산 그래프 | — | 복잡함수를 기본 연산으로 분해 |
| 국소 도함수 | — | 각 노드의 입력에 대한 출력 미분 |
| 역전파 | $\nabla_\theta \mathcal{L}$ | 1번 역전파로 모든 그래디언트 |
| 선형층 역전파 | $\partial\mathcal{L}/\partial W = \delta \mathbf{x}^\top$ | 가중치 그래디언트 |

## 연습문제

1. 미니 Autograd `Tensor` 클래스에 `__sub__`, `__truediv__`, `matmul` 연산을 추가하라.
2. 미니 Autograd로 $f(x, y) = (x + y)^2 - xy$의 $\partial f/\partial x, \partial f/\partial y$를 계산하고 수치 미분으로 검증하라.
3. PyTorch로 3층 MLP를 만들고, 한 스텝 역전파 후 `W1.grad`, `W2.grad`를 출력하라.
4. 깊이 1, 5, 10, 20층 MLP에서 gradient norm을 비교하여 그래디언트 소실을 실험으로 보이라.
5. He 초기화와 잘못된 초기화(단순 `randn`)를 비교하여 깊은 MLP 학습이 어떻게 달라지는지 실험하라.

> 해설: `solutions/ch07_solutions.ipynb`
